# AgentKit-ADK: five Google ADK agents (runs free on Colab)

Five end-to-end agent applications built with the **Google Agent Development Kit (ADK)**:

| | Agent | Pattern |
|---|---|---|
| A | Lead-generation deep research | `SequentialAgent` + `LoopAgent` (plan → research ⇄ critique → report) |
| B | Gemini CLI as a tool | planner → CLI executor → independent verifier |
| C | MCP bug assistant | `MCPToolset` over a local MCP server (SQLite + Stack Overflow + GitHub) |
| D | Code review assistant | AST analysis → ruff lint → sandboxed generated tests → fixed code |
| E | E-commerce agent | multi-agent + MCP + guardrails + transactional checkout |

**Before running:** add your free Gemini API key (https://aistudio.google.com/apikey) to Colab **Secrets** as `GOOGLE_API_KEY` and enable *Notebook access*.
Run the sections one at a time (not *Run all*) to stay inside the free-tier limits.

## 0. Setup

In [1]:
import os, shutil
os.chdir("/content")
shutil.rmtree("agentkit-adk", ignore_errors=True)
for d in ["a_lead_gen_agent", "b_gemini_cli_agent", "c_bug_assistant", "e_ecommerce_agent", "outputs",
          "d_code_review_assistant/samples", "d_code_review_assistant/tests"]:
    os.makedirs(f"agentkit-adk/{d}", exist_ok=True)
for d in ["a_lead_gen_agent", "b_gemini_cli_agent", "c_bug_assistant", "d_code_review_assistant", "e_ecommerce_agent"]:
    open(f"agentkit-adk/{d}/__init__.py", "w").write("from . import agent\n")
open("agentkit-adk/d_code_review_assistant/tests/__init__.py", "w").close()
os.chdir("/content/agentkit-adk")
print("working in", os.getcwd())
!pip install -q "google-adk>=1.5.0" "mcp>=1.9,<2" requests ruff pytest ddgs

working in /content/agentkit-adk


In [2]:
import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["ADK_MODEL"] = "gemini-3.5-flash-lite"      # free-tier friendly model for all agents
os.environ["GEMINI_CLI_MODEL"] = "gemini-3.8-flash"    # used only by agent B (Gemini CLI)
print("API key loaded:", bool(os.environ["GOOGLE_API_KEY"]))

API key loaded: True


### Shared helpers
`common.py` loads the key, runs agents and prints every step (🔧 tool call, ✅ tool result) with automatic retry on free-tier rate limits. `search_tool.py` is a free web-search tool. `run.py` is the entry point used below.

In [3]:
%%writefile common.py
"""Shared helpers: env/API-key loading + a tiny runner that prints every agent step.

Free setup: get a key at https://aistudio.google.com/apikey (no credit card) and
store it in Colab Secrets as GOOGLE_API_KEY (or export it as an env var).
"""
import os
import sys
import asyncio

from google.genai import types
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

# Flash-Lite = most generous free-tier daily limits (each agent makes many LLM calls).
# Gemini 2.5 models were retired in Oct 2026. Override with ADK_MODEL, e.g. "gemini-3.8-flash".
MODEL = os.getenv("ADK_MODEL", "gemini-3.5-flash-lite")
APP_NAME = "agentkit_adk"


def load_env() -> None:
    """Make sure GOOGLE_API_KEY is set (Colab Secrets -> env var fallback)."""
    os.environ.setdefault("GOOGLE_GENAI_USE_VERTEXAI", "FALSE")
    if not os.getenv("GOOGLE_API_KEY"):
        try:
            from google.colab import userdata  # only exists inside Colab
            os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
        except Exception:
            pass
    if not os.getenv("GOOGLE_API_KEY"):
        sys.exit("GOOGLE_API_KEY is not set. Get a free key at https://aistudio.google.com/apikey")
    # Gemini CLI (agent b) reads GEMINI_API_KEY
    os.environ.setdefault("GEMINI_API_KEY", os.environ["GOOGLE_API_KEY"])


def _short(obj, n: int = 220) -> str:
    s = str(obj).replace("\n", " ")
    return s if len(s) <= n else s[:n] + " ..."


async def _run_turn(runner, user_id, session_id, message, show_tools):
    content = types.Content(role="user", parts=[types.Part(text=message)])
    final = ""
    async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
        if show_tools:
            for fc in event.get_function_calls() or []:
                print(f"  🔧 [{event.author}] calls {fc.name}({_short(fc.args)})")
            for fr in event.get_function_responses() or []:
                print(f"  ✅ [{event.author}] got {fr.name} -> {_short(fr.response)}")
        if event.is_final_response() and event.content and event.content.parts:
            text = "".join(p.text or "" for p in event.content.parts).strip()
            if text:
                print(f"\n🤖 [{event.author}]\n{text}\n")
                final = text
    return final


async def chat(agent, messages, *, state=None, user_id="student", show_tools=True, retries=3):
    """Run one or more user turns in ONE session. Returns (last_answer, final_state)."""
    if isinstance(messages, str):
        messages = [messages]
    svc = InMemorySessionService()
    session = await svc.create_session(app_name=APP_NAME, user_id=user_id, state=state or {})
    runner = Runner(agent=agent, app_name=APP_NAME, session_service=svc)
    final = ""
    for msg in messages:
        print(f"\n🧑 USER: {msg}\n" + "-" * 60)
        for attempt in range(retries):
            try:
                final = await _run_turn(runner, user_id, session.id, msg, show_tools)
                break
            except Exception as e:  # free tier = rate limits; back off and retry
                if ("429" in str(e) or "RESOURCE_EXHAUSTED" in str(e)) and attempt < retries - 1:
                    wait = 30 * (attempt + 1)
                    print(f"⏳ Free-tier rate limit hit, retrying in {wait}s ...")
                    await asyncio.sleep(wait)
                else:
                    raise
    session = await svc.get_session(app_name=APP_NAME, user_id=user_id, session_id=session.id)
    return final, session.state

Writing common.py


In [4]:
%%writefile search_tool.py
"""Free web search tool (DuckDuckGo via `ddgs`): no API key, and it doesn't use Gemini's search-grounding quota."""
import time


def web_search(query: str) -> dict:
    """Search the web and return up to 8 results (title, url, snippet).

    Args:
        query: A short, specific search query.
    """
    from ddgs import DDGS  # imported lazily so agents still load if the package is missing
    last_err = ""
    for attempt in range(3):  # DuckDuckGo sometimes rate-limits; back off briefly
        try:
            hits = DDGS().text(query, max_results=8) or []
            return {"status": "success", "query": query,
                    "results": [{"title": h.get("title", ""), "url": h.get("href", ""),
                                 "snippet": (h.get("body", "") or "")[:300]} for h in hits]}
        except Exception as e:
            last_err = str(e)[:200]
            time.sleep(2 * (attempt + 1))
    return {"status": "error", "error": f"search failed: {last_err}. Try a different query."}

Writing search_tool.py


In [5]:
%%writefile run.py
"""One entry point for all five agents (Colab friendly).

    python run.py a            # lead generation deep research
    python run.py b            # Gemini CLI inside an ADK pipeline
    python run.py c            # MCP bug assistant (multi-turn)
    python run.py d [file.py]  # code review assistant
    python run.py e            # e-commerce agent (multi-turn)
    python run.py a --prompt "your own request"
"""
import argparse
import asyncio
import importlib
import os
import pathlib
import sys

from common import load_env, chat

DEMOS = {
    "a": ["Find 8 US B2B SaaS companies (50-300 employees, Series A-B) building developer observability or "
          "data-pipeline monitoring tools that are hiring data/platform engineers. We sell a data-quality testing tool."],
    "b": ["Create a small Python CLI todo app (add, list, done commands, saved to todos.json) with pytest tests. "
          "Run the tests and make sure they pass."],
    "c": ["Our checkout API returns 500 with 'QueuePool limit reached' during traffic spikes. Is this a known bug?",
          "Search Stack Overflow for fixes to SQLAlchemy connection pool exhaustion.",
          "Create a high severity bug in component 'database' titled 'Raise DB pool size and add pool_timeout' "
          "describing the checkout fix we discussed."],
    "d": None,  # reads samples/buggy_code.py
    "e": ["I need running shoes under $120. What do you recommend?",
          "Add the cheapest one to my cart, then show me my cart.",
          "Checkout please.",
          "Yes, place the order.",
          "What's the status of my order?"],
}
MODULES = {"a": "a_lead_gen_agent.agent", "b": "b_gemini_cli_agent.agent", "c": "c_bug_assistant.agent",
           "d": "d_code_review_assistant.agent", "e": "e_ecommerce_agent.agent"}


async def run_one(key: str, prompt: str | None, code_file: str | None):
    mod = importlib.import_module(MODULES[key])
    if key == "e":  # fresh DB so the demo is reproducible
        pathlib.Path("e_ecommerce_agent/shop.db").unlink(missing_ok=True)
    if key == "d":
        path = code_file or "d_code_review_assistant/samples/buggy_code.py"
        code = pathlib.Path(path).read_text()
        messages = [f"Please review this Python code:\n\n```python\n{code}\n```"]
    else:
        messages = [prompt] if prompt else DEMOS[key]
    try:
        answer, state = await chat(mod.root_agent, messages, state=dict(getattr(mod, "INITIAL_STATE", {})))
    finally:
        for ts in getattr(mod, "TOOLSETS", []):  # close MCP subprocesses
            try:
                await ts.close()
            except Exception:
                pass
    print("=" * 60, f"\n✔ Agent {key} finished. Output files are in ./{os.getenv('OUTPUT_DIR', 'outputs')}/")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("agent", choices=[*MODULES, "all"])
    ap.add_argument("file", nargs="?", help="python file to review (agent d only)")
    ap.add_argument("--prompt", help="custom prompt (single turn)")
    a = ap.parse_args()
    load_env()
    for k in (MODULES if a.agent == "all" else [a.agent]):
        try:
            asyncio.run(run_one(k, a.prompt, a.file))
        except Exception as e:  # short, readable error instead of a 100-line traceback
            root = e
            while getattr(root, "__cause__", None) or (getattr(root, "exceptions", None) and root.exceptions):
                root = root.__cause__ or root.exceptions[0]
            print(f"\n❌ AGENT {k} FAILED: {type(root).__name__}: {str(root)[:1200]}")
            if "NOT_FOUND" in str(root) or "404" in str(root):
                print("💡 Model not available for your key. List usable models with the snippet in the README, "
                      "then set os.environ['ADK_MODEL'] = '<model id>' before running.")
            sys.exit(1)

Writing run.py


## A. Lead-generation deep research agent
`research_planner → LoopAgent[lead_researcher ⇄ research_evaluator] → lead_report_writer`

The planner builds an ideal-customer profile, search queries and a scoring rubric. The researcher gathers leads with web search; the evaluator checks completeness and either sends feedback back or ends the loop (`exit_loop` sets `escalate`). The writer scores the leads and saves a CSV + report.

In [6]:
%%writefile a_lead_gen_agent/agent.py
"""(a) Deep-research lead-generation agent.

Pipeline:  planner -> [researcher <-> evaluator] x N (LoopAgent) -> report writer
Pattern follows Google's "deep research agent with ADK" blog: plan, search, critique, refine, write.
Only public, company-level info is collected (no personal emails/phones).
"""
import os
import pathlib

from google.adk.agents import LlmAgent, SequentialAgent, LoopAgent
from google.adk.tools.tool_context import ToolContext

from common import MODEL
from search_tool import web_search

OUT_DIR = pathlib.Path(os.getenv("OUTPUT_DIR", "outputs"))
INITIAL_STATE = {"raw_leads": "(none yet)", "evaluation": "(first pass)"}


# ---------- tools ----------
def exit_loop(tool_context: ToolContext) -> dict:
    """Call this when the lead list is complete enough to stop researching."""
    tool_context.actions.escalate = True  # tells the LoopAgent to stop
    return {"status": "research complete"}


def save_output(filename: str, content: str) -> dict:
    """Save text content (CSV or Markdown) to the outputs folder. filename like 'leads.csv'."""
    OUT_DIR.mkdir(exist_ok=True)
    safe = pathlib.Path(filename).name  # block path traversal
    (OUT_DIR / safe).write_text(content, encoding="utf-8")
    return {"status": "saved", "path": str(OUT_DIR / safe)}


# ---------- agents ----------
planner = LlmAgent(
    name="research_planner",
    model=MODEL,
    description="Turns the user's lead request into an ICP + search plan.",
    instruction="""You plan B2B lead-generation research.
From the user's request produce:
1. ICP: industry, company size, geography, funding stage, buying signals.
2. 5 distinct Google search queries (different angles: product category, hiring signals,
   funding news, tech stack, competitor customers).
3. A 100-point scoring rubric (fit 40, buying signal 30, reachability 20, data confidence 10).
Keep it under 200 words.""",
    output_key="research_plan",
)

researcher = LlmAgent(
    name="lead_researcher",
    model=MODEL,
    description="Searches the web and compiles candidate leads.",
    tools=[web_search],
    instruction="""You are a lead researcher. Use the web_search tool (several different queries) to execute this plan:
{research_plan}

Leads found so far:
{raw_leads?}

Reviewer feedback to address:
{evaluation?}

Find 8-10 distinct companies that match the ICP. For EACH give:
company, website, HQ location, approx size, what they do, ONE concrete buying signal
(hiring, funding, launch) with its source URL, and the best-fit DECISION-MAKER ROLE
(a job title only - never personal emails or phone numbers).
Output the full consolidated list (previous + new), no commentary.""",
    output_key="raw_leads",
)

evaluator = LlmAgent(
    name="research_evaluator",
    model=MODEL,
    description="Critiques the lead list and decides whether to loop again.",
    tools=[exit_loop],
    instruction="""You are a strict research reviewer. Check these leads:
{raw_leads}

Pass criteria: at least 8 companies; each has website, size, a sourced buying signal, a target role;
no duplicates. If ALL pass, call the exit_loop tool and reply 'APPROVED'.
Otherwise reply with a short bullet list of exactly what is missing so the researcher can fix it.""",
    output_key="evaluation",
)

research_loop = LoopAgent(
    name="research_loop",
    sub_agents=[researcher, evaluator],
    max_iterations=2,  # keeps Colab runs fast + inside free-tier limits
)

writer = LlmAgent(
    name="lead_report_writer",
    model=MODEL,
    description="Scores leads and writes the final deliverables.",
    tools=[save_output],
    instruction="""Using the plan {research_plan} and leads {raw_leads}:
1. Score every lead 0-100 with the rubric and sort high to low.
2. Call save_output('leads.csv', <CSV text: rank,company,website,score,signal,target_role,outreach_angle>).
3. Call save_output('lead_report.md', <markdown report>).
4. Reply with the final ranked markdown table (top 10) and a 2-line outreach angle for the top 3.""",
    output_key="final_report",
)

root_agent = SequentialAgent(
    name="lead_gen_deep_research",
    sub_agents=[planner, research_loop, writer],
)

Writing a_lead_gen_agent/agent.py


In [7]:
!python run.py a


🧑 USER: Find 8 US B2B SaaS companies (50-300 employees, Series A-B) building developer observability or data-pipeline monitoring tools that are hiring data/platform engineers. We sell a data-quality testing tool.
------------------------------------------------------------
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.

🤖 [research_planner]
**1. ICP**
* **Industry:** B2B SaaS (Developer Observability & Data-Pipeline Monitoring)
* **Company Size:** 50-300 employees
* **Geography:** United States
* **Funding Stage:** Series A - Series B
* **Buying Signals:** Actively hiring data/platform engineers, expanding data ingestion infrastructure, building monitoring/alerting features.

**2. Google Search Queries**
* *Product Category:* `"data-pipeline monitoring" OR "developer observability" startup US site:linkedin.com/company`
* *Hiring Signals:* `B2B SaaS "data engineer" OR "platform engineer" hiring Series A OR Series B US`
* *Funding News:* `"Series A" OR "Series B" 

In [8]:
!ls outputs
!head -6 outputs/leads.csv

lead_report.md	leads.csv
rank,company,website,score,signal,target_role,outreach_angle
1,Metaplane,https://www.metaplane.dev,95,Actively hiring platform/data engineers & scaling observability workflows,VP of Engineering,Embed automated data-quality testing directly into Metaplane's pipeline monitoring to reduce false positives.
2,Telmai,https://www.telmai.com,90,Expanding streaming data ingestion architecture & hiring infrastructure engineers,VP of Engineering,Enhance streaming data profiling in Telmai with lightweight, developer-first quality tests.
3,Masthead Data,https://www.mastheaddata.io,88,Hiring platform engineers to support expanded cloud data warehouse integrations,Head of Product,Scale metadata-driven anomaly detection with proactive data quality checks.
4,Anomalo,https://anomalo.com,85,Expanding automated alerting and hiring distributed systems/data engineers,Head of Product,Complement ML anomaly detection with explicit data quality assertions for engineering teams.
5,Soda,h

## B. Gemini CLI as a tool inside an ADK pipeline
`task_planner → cli_executor (gemini -p …) → verifier`

The CLI runs non-interactively in a sandbox folder. The verifier does **not** trust the CLI's text: it lists the real files and re-runs pytest itself.

In [9]:
%%writefile b_gemini_cli_agent/agent.py
"""(b) Advanced tool agent: Gemini CLI wrapped as an ADK tool inside a pipeline.

Pipeline: task_planner -> cli_executor (runs `gemini -p ...`) -> verifier (reads the files it made)
Gemini CLI is installed with:  npm install -g @google/gemini-cli   (needs Node 20+)
"""
import os
import pathlib
import shutil
import subprocess
import sys

from google.adk.agents import LlmAgent, SequentialAgent

from common import MODEL

WORKSPACE = pathlib.Path(os.getenv("GEMINI_WORKSPACE", "workspace")).resolve()
CLI_MODEL = os.getenv("GEMINI_CLI_MODEL", "gemini-3.8-flash")  # stronger model: tool use is unreliable on Lite
CLI_TIMEOUT = int(os.getenv("GEMINI_CLI_TIMEOUT", "240"))
INITIAL_STATE = {}


def run_gemini_cli(prompt: str) -> dict:
    """Run Gemini CLI non-interactively inside the sandbox workspace and return its output.

    Args:
        prompt: A complete, self-contained instruction for the Gemini CLI coding agent.
    """
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    if not shutil.which("gemini"):
        return {"status": "error", "error": "gemini CLI not found. Run: npm install -g @google/gemini-cli"}
    cmd = ["gemini", "-p", prompt, "-m", CLI_MODEL]
    if os.getenv("GEMINI_YOLO", "1") == "1":  # auto-approve file writes/shell inside the workspace
        cmd.append("--yolo")
    try:
        r = subprocess.run(cmd, cwd=WORKSPACE, capture_output=True, text=True,
                           timeout=CLI_TIMEOUT, env=os.environ.copy())
    except subprocess.TimeoutExpired:
        return {"status": "error", "error": f"gemini CLI timed out after {CLI_TIMEOUT}s"}
    files = sorted(str(p.relative_to(WORKSPACE)) for p in WORKSPACE.rglob("*") if p.is_file())
    status = "success" if r.returncode == 0 else "error"
    note = ""
    if status == "success" and not files:  # CLI may just describe work (or invent test output) without doing it
        status = "no_files_created"
        note = "Gemini CLI exited normally but the workspace is EMPTY. Its text output is NOT proof of work."
    return {"status": status, "note": note, "stdout": r.stdout[-4000:], "stderr": r.stderr[-1500:], "files_now": files}


def list_workspace_files() -> dict:
    """List the files currently in the sandbox workspace."""
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    return {"files": sorted(str(p.relative_to(WORKSPACE)) for p in WORKSPACE.rglob("*") if p.is_file())}


def read_workspace_file(path: str) -> dict:
    """Read a text file from the sandbox workspace.

    Args:
        path: Relative path inside the workspace.
    """
    target = (WORKSPACE / path).resolve()
    if WORKSPACE not in target.parents and target != WORKSPACE:  # path-traversal guard
        return {"status": "error", "error": "path escapes workspace"}
    if not target.is_file():
        return {"status": "error", "error": "file not found"}
    return {"status": "success", "content": target.read_text(errors="replace")[:8000]}


def run_workspace_tests() -> dict:
    """Run pytest inside the workspace and return the REAL result (use this instead of trusting the CLI's text)."""
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    try:
        r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--disable-warnings", "-p", "no:cacheprovider"],
                           cwd=WORKSPACE, capture_output=True, text=True, timeout=60)
    except subprocess.TimeoutExpired:
        return {"status": "error", "error": "pytest timed out"}
    return {"status": "passed" if r.returncode == 0 else "failed", "output": (r.stdout + r.stderr)[-2500:]}


planner = LlmAgent(
    name="task_planner",
    model=MODEL,
    description="Rewrites the user's request into one precise prompt for Gemini CLI.",
    instruction="""Turn the user's request into ONE clear, self-contained prompt for a coding CLI agent.
State: files to create, language, behaviour, tests, and 'run the tests and fix failures'.
Explicitly say: "Use your file-writing and shell tools to actually create the files in the current directory
and run the tests with python -m pytest. Do not just print code." Output only the prompt text.""",
    output_key="cli_prompt",
)

executor = LlmAgent(
    name="cli_executor",
    model=MODEL,
    description="Runs Gemini CLI via the tool.",
    tools=[run_gemini_cli],
    instruction="""Call run_gemini_cli exactly once with this prompt:
{cli_prompt}
Report the tool's status honestly. If status is 'error' or 'no_files_created', say so plainly and do NOT claim success.
Otherwise summarise what the CLI said and list files_now.""",
    output_key="cli_result",
)

verifier = LlmAgent(
    name="verifier",
    model=MODEL,
    description="Independently checks the CLI's work using real files and real test runs.",
    tools=[list_workspace_files, read_workspace_file, run_workspace_tests],
    instruction="""Original goal prompt: {cli_prompt}
CLI result: {cli_result}

Do NOT trust the CLI's text (it can invent files and test output). Verify yourself, each tool once unless state changes:
1. list_workspace_files. If it is empty -> the verdict is FAIL (nothing was created), stop.
2. read_workspace_file for the main files.
3. run_workspace_tests and use ONLY its result.
Verdict rules: PASS only if the expected files exist AND run_workspace_tests status is 'passed'. Otherwise FAIL.
Reply: PASS or FAIL, a 3-bullet evidence summary (quote the real tool results), and suggested fixes.""",
)

root_agent = SequentialAgent(name="gemini_cli_pipeline", sub_agents=[planner, executor, verifier])

Writing b_gemini_cli_agent/agent.py


In [10]:
!node --version
!npm install -g @google/gemini-cli > /dev/null 2>&1; gemini --version

v22.23.4
0.66.0


In [11]:
!rm -rf workspace
!python run.py b
!ls workspace


🧑 USER: Create a small Python CLI todo app (add, list, done commands, saved to todos.json) with pytest tests. Run the tests and make sure they pass.
------------------------------------------------------------
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.

🤖 [task_planner]
Create a Python CLI todo application in the current directory.

Files to create:
1. `todo.py`: A CLI application using `argparse` with three commands:
   - `add <task>`: Adds a new task to `todos.json`. Each task should be a dictionary with `id`, `task`, and `completed` (default `False`).
   - `list`: Lists all tasks with their ID, completion status, and task description.
   - `done <id>`: Marks the task with the given ID as completed (`completed=True`) in `todos.json`.
   Data must persist in `todos.json` in the current directory. Handle missing files gracefully (e.g., return empty list).

2. `test_todo.py`: A comprehensive `pytest` test suite covering:
   - Adding a task and verifying it ap

## C. MCP-tools bug assistant
`software_bug_assistant` ⟷ local MCP server (SQLite bug tracker + Stack Overflow + GitHub issues)

`MCPToolset` launches the server over stdio and converts its tools into ADK tools. The demo is a 3-turn conversation: triage a bug, research fixes, then file a new bug.

In [12]:
%%writefile c_bug_assistant/bug_mcp_server.py
"""Local MCP server (stdio) for the bug assistant. 100% free: SQLite + public Stack Exchange/GitHub APIs.
NEVER print to stdout here - stdout is the MCP protocol channel. Log to stderr only.
"""
import os
import sqlite3
import sys
from pathlib import Path

import requests
from mcp.server.fastmcp import FastMCP

DB = Path(os.getenv("BUG_DB", Path(__file__).with_name("bugs.db")))
mcp = FastMCP("bug-tracker")

SEED = [
    ("Connection pool exhausted under load", "Checkout API returns 500 with 'QueuePool limit reached' when traffic spikes.", "high", "open", "", "database"),
    ("NullPointerException in OrderService.calculateTotal", "NPE when a cart item has no discount object.", "high", "resolved", "Added null-safe default discount; released in v2.3.1.", "backend"),
    ("KeyError loading config in Python worker", "Worker crashes when optional key 'retry_policy' is missing.", "medium", "resolved", "Use config.get('retry_policy', DEFAULTS).", "python"),
    ("Login redirect loop on Safari", "ITP blocks third-party cookie so OAuth callback loops.", "medium", "open", "", "frontend"),
    ("Memory leak in websocket handler", "Heap grows ~50MB/hour; listeners are never removed on disconnect.", "critical", "in_progress", "", "backend"),
    ("Timeout calling payment provider", "Intermittent 504 from provider; no retry/backoff in client.", "high", "resolved", "Added exponential backoff (3 tries) + idempotency key.", "payments"),
    ("CSV export has wrong encoding", "Non-ASCII names show as '?' in Excel.", "low", "resolved", "Write UTF-8 with BOM.", "reporting"),
]


def _db() -> sqlite3.Connection:
    first = not DB.exists()
    con = sqlite3.connect(DB)
    con.row_factory = sqlite3.Row
    if first:
        con.execute("""CREATE TABLE bugs(id INTEGER PRIMARY KEY AUTOINCREMENT, title TEXT, description TEXT,
                       severity TEXT, status TEXT, resolution TEXT, component TEXT)""")
        con.executemany("INSERT INTO bugs(title,description,severity,status,resolution,component) VALUES(?,?,?,?,?,?)", SEED)
        con.commit()
    return con


@mcp.tool()
def search_bugs(query: str, status: str = "") -> list[dict]:
    """Search the internal bug tracker by keyword in title/description. Optional status: open, in_progress, resolved."""
    words = [w for w in query.lower().split() if len(w) > 2] or [query.lower()]
    sql = "SELECT * FROM bugs WHERE (" + " OR ".join(["lower(title||' '||description) LIKE ?"] * len(words)) + ")"
    args = [f"%{w}%" for w in words]
    if status:
        sql += " AND status = ?"
        args.append(status)
    with _db() as con:
        return [dict(r) for r in con.execute(sql + " LIMIT 10", args)]


@mcp.tool()
def get_bug(bug_id: int) -> dict:
    """Get one bug by id."""
    with _db() as con:
        row = con.execute("SELECT * FROM bugs WHERE id=?", (bug_id,)).fetchone()
    return dict(row) if row else {"error": f"bug {bug_id} not found"}


@mcp.tool()
def create_bug(title: str, description: str, severity: str, component: str = "general") -> dict:
    """Create a new bug. severity must be one of: low, medium, high, critical."""
    if severity not in {"low", "medium", "high", "critical"}:
        return {"error": "severity must be low, medium, high or critical"}
    with _db() as con:
        cur = con.execute("INSERT INTO bugs(title,description,severity,status,resolution,component) VALUES(?,?,?,?,?,?)",
                          (title, description, severity, "open", "", component))
        return {"created_id": cur.lastrowid, "status": "open"}


@mcp.tool()
def update_bug_status(bug_id: int, status: str, resolution: str = "") -> dict:
    """Update a bug's status (open, in_progress, resolved) and optionally record the resolution."""
    if status not in {"open", "in_progress", "resolved"}:
        return {"error": "invalid status"}
    with _db() as con:
        cur = con.execute("UPDATE bugs SET status=?, resolution=? WHERE id=?", (status, resolution, bug_id))
        return {"updated": cur.rowcount == 1}


@mcp.tool()
def search_stackoverflow(query: str) -> list[dict]:
    """Search Stack Overflow (free public API) for questions that have an accepted answer."""
    try:
        r = requests.get("https://api.stackexchange.com/2.3/search/advanced", timeout=15,
                         params={"order": "desc", "sort": "relevance", "q": query, "accepted": "True",
                                 "site": "stackoverflow", "pagesize": 5})
        r.raise_for_status()
        return [{"title": i["title"], "link": i["link"], "score": i["score"], "tags": i["tags"]}
                for i in r.json().get("items", [])]
    except Exception as e:
        return [{"error": f"stackoverflow lookup failed: {e}"}]


@mcp.tool()
def search_github_issues(repo: str, query: str) -> list[dict]:
    """Search issues of a public GitHub repo, e.g. repo='sqlalchemy/sqlalchemy'."""
    headers = {"Accept": "application/vnd.github+json"}
    if os.getenv("GITHUB_TOKEN"):
        headers["Authorization"] = f"Bearer {os.environ['GITHUB_TOKEN']}"
    try:
        r = requests.get("https://api.github.com/search/issues", headers=headers, timeout=15,
                         params={"q": f"{query} repo:{repo} is:issue", "per_page": 5})
        r.raise_for_status()
        return [{"title": i["title"], "url": i["html_url"], "state": i["state"]} for i in r.json().get("items", [])]
    except Exception as e:
        return [{"error": f"github lookup failed: {e}"}]


if __name__ == "__main__":
    print("bug-tracker MCP server starting", file=sys.stderr)
    mcp.run()  # stdio transport

Writing c_bug_assistant/bug_mcp_server.py


In [13]:
%%writefile c_bug_assistant/agent.py
"""(c) MCP-tools bug assistant.

ADK agent  --MCP(stdio)-->  bug_mcp_server.py  (SQLite bug DB + Stack Overflow + GitHub issues)
Plus a free DuckDuckGo web_search function tool for docs/release notes.
"""
import sys
from pathlib import Path

from google.adk.agents import LlmAgent
from google.adk.tools.mcp_tool.mcp_toolset import MCPToolset, StdioConnectionParams
from mcp import StdioServerParameters

from common import MODEL
from search_tool import web_search

SERVER = str(Path(__file__).with_name("bug_mcp_server.py"))
INITIAL_STATE = {}

bug_tools = MCPToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(command=sys.executable, args=[SERVER]),
        timeout=30,
    )
)
TOOLSETS = [bug_tools]  # run.py closes these so the MCP subprocess exits cleanly

root_agent = LlmAgent(
    name="software_bug_assistant",
    model=MODEL,
    description="Helps engineers triage, research and track software bugs.",
    instruction="""You are a senior support engineer helping triage bugs.
Workflow for every bug report:
1. search_bugs in the internal tracker first - mention matching bug ids and known resolutions.
2. If no fix is known, search_stackoverflow (and search_github_issues if a library/repo is named).
3. Use web_search only for current docs/release notes.
4. Only call create_bug or update_bug_status when the user asks you to; confirm the id afterwards.
Be concise: root cause hypothesis, evidence with links, recommended next step.""",
    tools=[bug_tools, web_search],
)

Writing c_bug_assistant/agent.py


In [14]:
!python run.py c

/usr/local/lib/python3.13/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()

🧑 USER: Our checkout API returns 500 with 'QueuePool limit reached' during traffic spikes. Is this a known bug?
------------------------------------------------------------
/usr/local/lib/python3.13/dist-packages/google/adk/tools/mcp_tool/mcp_toolset.py:459: UserWarning: [EXPERIMENTAL] feature FeatureName._MCP_GRACEFUL_ERROR_HANDLING is enabled.
  session = await self._mcp_session_manager.create_session(
bug-tracker MCP server starting
[10/27/26 23:15:14] INFO     Processing request of type            ]8;id=992425;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=77805;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/server.py#733\733]8;;\
                             ListToolsRequest                                   
/usr/loca

## D. Code review assistant
`structure_analyzer → style_checker → test_runner → review_synthesizer`

Deterministic tools find the facts (AST analysis, ruff lint, sandboxed pytest with a timeout and no API keys in the environment); the LLM explains and writes the fixed code.

In [15]:
%%writefile d_code_review_assistant/tools.py
"""Deterministic tools for the code-review pipeline (AST analysis, ruff linting, sandboxed pytest)."""
import ast
import json
import logging
import os
import pathlib
import subprocess
import sys
import tempfile

log = logging.getLogger("code_review")
OUT_DIR = pathlib.Path(os.getenv("OUTPUT_DIR", "outputs"))
MAX_CODE_CHARS = 20_000  # input guardrail


def _complexity(node: ast.AST) -> int:
    score = 1
    for n in ast.walk(node):
        if isinstance(n, (ast.If, ast.For, ast.While, ast.ExceptHandler, ast.IfExp, ast.comprehension, ast.AsyncFor)):
            score += 1
        elif isinstance(n, ast.BoolOp):
            score += len(n.values) - 1
    return score


def analyze_code_structure(code: str) -> dict:
    """Parse Python code with the AST. Returns functions, classes, complexity and risky patterns.

    Args:
        code: The full Python source to analyze.
    """
    if len(code) > MAX_CODE_CHARS:
        return {"status": "error", "error": f"code too large (>{MAX_CODE_CHARS} chars)"}
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return {"status": "error", "error": f"SyntaxError line {e.lineno}: {e.msg}"}

    functions, classes, risks = [], [], []
    for n in ast.walk(tree):
        if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)):
            functions.append({
                "name": n.name, "line": n.lineno, "args": [a.arg for a in n.args.args],
                "lines": (n.end_lineno or n.lineno) - n.lineno + 1, "complexity": _complexity(n),
                "has_docstring": ast.get_docstring(n) is not None, "has_return_annotation": n.returns is not None,
            })
            for d in n.args.defaults:
                if isinstance(d, (ast.List, ast.Dict, ast.Set)):
                    risks.append({"line": n.lineno, "issue": f"mutable default argument in {n.name}()"})
        elif isinstance(n, ast.ClassDef):
            classes.append({"name": n.name, "line": n.lineno, "methods": sum(isinstance(b, ast.FunctionDef) for b in n.body)})
        elif isinstance(n, ast.ExceptHandler) and n.type is None:
            risks.append({"line": n.lineno, "issue": "bare except swallows all errors"})
        elif isinstance(n, ast.Call):
            fn = getattr(n.func, "id", getattr(n.func, "attr", ""))
            if fn in {"eval", "exec"}:
                risks.append({"line": n.lineno, "issue": f"use of {fn}() - code injection risk"})
            for kw in n.keywords:
                if kw.arg == "shell" and getattr(kw.value, "value", False) is True:
                    risks.append({"line": n.lineno, "issue": "subprocess with shell=True - command injection risk"})
    return {"status": "success", "functions": functions, "classes": classes, "risky_patterns": risks,
            "max_complexity": max((f["complexity"] for f in functions), default=0)}


def check_style(code: str) -> dict:
    """Lint Python code with ruff (pycodestyle, pyflakes, bugbear, bandit security rules).

    Args:
        code: The full Python source to lint.
    """
    if len(code) > MAX_CODE_CHARS:
        return {"status": "error", "error": "code too large"}
    try:
        r = subprocess.run([sys.executable, "-m", "ruff", "check", "--isolated", "--select", "E,F,W,B,S",
                            "--output-format", "json", "--stdin-filename", "submitted.py", "-"],
                           input=code, capture_output=True, text=True, timeout=30)
        issues = json.loads(r.stdout or "[]")
    except Exception as e:
        log.exception("ruff failed")
        return {"status": "error", "error": f"ruff unavailable: {e}"}
    return {"status": "success", "issue_count": len(issues),
            "issues": [{"rule": i["code"], "line": i["location"]["row"], "message": i["message"]} for i in issues[:25]]}


def run_generated_tests(code: str, tests: str) -> dict:
    """Run pytest tests against the submitted code in an isolated temp dir (API keys stripped, 30s timeout).

    Args:
        code: The code under test (saved as solution.py).
        tests: A pytest file. It must start with 'from solution import *'.
    """
    with tempfile.TemporaryDirectory() as d:
        pathlib.Path(d, "solution.py").write_text(code)
        pathlib.Path(d, "test_solution.py").write_text(tests)
        env = {"PATH": os.environ.get("PATH", ""), "PYTHONDONTWRITEBYTECODE": "1"}  # no secrets leak to test code
        try:
            r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--disable-warnings", "-p", "no:cacheprovider"],
                               cwd=d, capture_output=True, text=True, timeout=30, env=env)
        except subprocess.TimeoutExpired:
            return {"status": "error", "error": "tests timed out after 30s"}
    return {"status": "passed" if r.returncode == 0 else "failed", "output": (r.stdout + r.stderr)[-3000:]}


def save_review(content: str) -> dict:
    """Save the final markdown review to outputs/code_review.md.

    Args:
        content: The markdown review text.
    """
    OUT_DIR.mkdir(exist_ok=True)
    path = OUT_DIR / "code_review.md"
    path.write_text(content, encoding="utf-8")
    return {"status": "saved", "path": str(path)}

Writing d_code_review_assistant/tools.py


In [16]:
%%writefile d_code_review_assistant/agent.py
"""(d) Production-style code review assistant (SequentialAgent pipeline).

structure_analyzer (AST) -> style_checker (ruff) -> test_runner (LLM-written pytest, sandboxed) -> review_synthesizer
Production touches: deterministic tools do the facts, LLM only explains/fixes; input size limits; timeouts;
secrets stripped from the test sandbox; unit tests for every tool; logs; saved report artifact.
"""
from google.adk.agents import LlmAgent, SequentialAgent

from common import MODEL
from .tools import analyze_code_structure, check_style, run_generated_tests, save_review

INITIAL_STATE = {}

structure_analyzer = LlmAgent(
    name="structure_analyzer",
    model=MODEL,
    description="Analyses code structure and risky patterns.",
    tools=[analyze_code_structure],
    instruction="""The user's Python code is in the conversation. Call analyze_code_structure with the FULL code
(unchanged). Summarise: functions, complexity hot-spots (>8), missing docstrings, and each risky pattern with line numbers.""",
    output_key="structure_report",
)

style_checker = LlmAgent(
    name="style_checker",
    model=MODEL,
    description="Lints the code.",
    tools=[check_style],
    instruction="""Call check_style with the FULL user code. Group the findings into Security, Bugs, Style. Be brief.""",
    output_key="style_report",
)

test_runner = LlmAgent(
    name="test_runner",
    model=MODEL,
    description="Writes and runs pytest tests.",
    tools=[run_generated_tests],
    instruction="""Write 4-6 focused pytest tests (start the file with 'from solution import *') covering normal cases,
edge cases and the bugs suspected in: {structure_report}
Call run_generated_tests(code=<the FULL user code>, tests=<your tests>). Report which tests pass/fail and why.
Failures that expose real bugs are valuable - do not hide them.""",
    output_key="test_report",
)

review_synthesizer = LlmAgent(
    name="review_synthesizer",
    model=MODEL,
    description="Combines everything into the final review and a fixed version.",
    tools=[save_review],
    instruction="""Write the final code review using:
Structure: {structure_report}
Style: {style_report}
Tests: {test_report}

Format (markdown): 1) Verdict (APPROVE / CHANGES REQUESTED) 2) Findings table (severity, line, issue, fix)
3) Corrected code in one python block 4) Suggested tests worth keeping.
Call save_review with the full markdown, then reply with the review.""",
    output_key="final_review",
)

root_agent = SequentialAgent(
    name="code_review_assistant",
    sub_agents=[structure_analyzer, style_checker, test_runner, review_synthesizer],
)

Writing d_code_review_assistant/agent.py


In [17]:
%%writefile d_code_review_assistant/samples/buggy_code.py
import subprocess


def add_item(item, cart=[]):
    cart.append(item)
    return cart


def average_price(prices):
    total = 0
    for p in prices:
        total = total + p
    return total / len(prices)


def run_report(name):
    try:
        subprocess.run("echo generating " + name, shell=True)
    except:
        pass


def calc(expr):
    return eval(expr)

Writing d_code_review_assistant/samples/buggy_code.py


In [18]:
%%writefile d_code_review_assistant/tests/test_tools.py
from d_code_review_assistant.tools import analyze_code_structure, check_style, run_generated_tests

BAD = "def f(x, y=[]):\n    try:\n        return eval(x)\n    except:\n        pass\n"


def test_structure_flags_risks():
    r = analyze_code_structure(BAD)
    issues = " ".join(x["issue"] for x in r["risky_patterns"])
    assert "mutable default" in issues and "bare except" in issues and "eval" in issues


def test_syntax_error_is_handled():
    assert analyze_code_structure("def (")["status"] == "error"


def test_style_runs():
    r = check_style(BAD)
    assert r["status"] == "success" and r["issue_count"] > 0


def test_sandbox_runs_tests():
    ok = run_generated_tests("def add(a, b):\n    return a + b\n", "from solution import *\ndef test_a():\n    assert add(1, 2) == 3\n")
    bad = run_generated_tests("def add(a, b):\n    return a - b\n", "from solution import *\ndef test_a():\n    assert add(1, 2) == 3\n")
    assert ok["status"] == "passed" and bad["status"] == "failed"

Writing d_code_review_assistant/tests/test_tools.py


In [19]:
!pytest -q d_code_review_assistant/tests

....                                                                     [100%]
4 passed in 6.42s


In [20]:
!python run.py d


🧑 USER: Please review this Python code:

```python
import subprocess


def add_item(item, cart=[]):
    cart.append(item)
    return cart


def average_price(prices):
    total = 0
    for p in prices:
        total = total + p
    return total / len(prices)


def run_report(name):
    try:
        subprocess.run("echo generating " + name, shell=True)
    except:
        pass


def calc(expr):
    return eval(expr)

```
------------------------------------------------------------
/usr/local/lib/python3.13/dist-packages/google/adk/models/llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
  🔧 [structure_analyzer] calls analyze_code_structure({'code': 'import subprocess\n\n\ndef add_item(item, cart=[]):\n    cart.append(item)\n    return cart\n\n\ndef average_price(prices):\n    total = 0\n    for p in prices:\n        total 

In [21]:
!cat outputs/code_review.md

### **Verdict**
**CHANGES REQUESTED**

---

### **Findings Table**

| Severity | Line | Issue | Fix |
| :--- | :--- | :--- | :--- |
| **High (Security)** | 18 | `subprocess` with `shell=True` and string concatenation allows command injection. | Use `subprocess.run(["echo", f"generating {name}"], shell=False)` |
| **High (Security)** | 24 | Use of `eval()` allows arbitrary code execution (RCE). | Use `ast.literal_eval()` or a safe expression parser. |
| **Medium (Bug)** | 4 | Mutable default argument (`cart=[]`) shares state across function calls. | Use `None` as default and initialize `cart = []` inside the function. |
| **Medium (Bug)** | 19 | Bare `except:` swallows all errors (including `KeyboardInterrupt`). | Catch specific exceptions like `(subprocess.SubprocessError, OSError)`. |
| **Low (Bug/Robustness)**| 9 | Potential `ZeroDivisionError` if `prices` list is empty. | Check if `prices` is empty before dividing. |

---

### **Corrected Code**

```python
import ast
import subproce

## E. E-commerce agent
`shopping_agent` → transfer → `order_support_agent`, both on a SQLite-backed MCP server

Guardrails in a `before_tool_callback`: the user ID is injected from session state, and checkout is blocked until the user confirms. Checkout/cancel are atomic DB transactions.

In [26]:
%%writefile e_ecommerce_agent/mcp_server.py
"""E-commerce MCP server (stdio) backed by SQLite - the free stand-in for AlloyDB + MCP Toolbox.
Transactional checkout (stock check + order + cart clear happen atomically). Log to stderr only.
"""
import os
import sqlite3
import sys
from pathlib import Path

from mcp.server.fastmcp import FastMCP

DB = Path(os.getenv("SHOP_DB", Path(__file__).with_name("shop.db")))
mcp = FastMCP("ecommerce-store")

PRODUCTS = [  # name, category, price, stock, description
    ("AeroRun 2 Running Shoes", "shoes", 99.0, 25, "Lightweight daily trainer with breathable mesh"),
    ("TrailBlazer GTX Trail Shoes", "shoes", 139.0, 12, "Waterproof trail running shoe, aggressive grip"),
    ("SprintX Racing Flats", "shoes", 119.0, 8, "Carbon-plated race shoe for 5K to half marathon"),
    ("CloudWalk Running Shoes", "shoes", 79.0, 40, "Budget cushioned running shoe for beginners"),
    ("Court Pro Basketball Shoes", "shoes", 109.0, 15, "High-top basketball shoe with ankle support"),
    ("PulseBuds Wireless Earbuds", "audio", 59.0, 50, "Bluetooth 5.3 earbuds, 24h battery"),
    ("StudioMax Over-Ear Headphones", "audio", 149.0, 18, "Active noise cancelling, 40h battery"),
    ("BassBox Portable Speaker", "audio", 45.0, 30, "Waterproof Bluetooth speaker"),
    ("FitBand 5 Fitness Tracker", "wearables", 49.0, 60, "Heart rate, sleep and step tracking"),
    ("Chrono Smartwatch S", "wearables", 199.0, 14, "GPS smartwatch with AMOLED display"),
    ("HydroFlask 1L Bottle", "outdoor", 29.0, 80, "Insulated steel bottle keeps drinks cold 24h"),
    ("Summit 40L Backpack", "outdoor", 89.0, 22, "Hiking backpack with rain cover"),
    ("NightOwl Headlamp", "outdoor", 25.0, 45, "Rechargeable 600-lumen headlamp"),
    ("ErgoDesk Laptop Stand", "office", 35.0, 33, "Adjustable aluminium laptop stand"),
    ("SilentKey Mechanical Keyboard", "office", 85.0, 20, "Hot-swappable silent switches"),
    ("GlideMouse Wireless Mouse", "office", 29.0, 70, "Ergonomic wireless mouse"),
    ("PowerBrick 20k Power Bank", "accessories", 39.0, 55, "20000mAh USB-C fast charge"),
    ("FlexCable 3-Pack USB-C", "accessories", 15.0, 100, "Braided 1m USB-C cables"),
    ("YogaFlow Mat", "fitness", 32.0, 40, "6mm non-slip yoga mat"),
    ("IronGrip Adjustable Dumbbell", "fitness", 129.0, 9, "5-52 lb adjustable dumbbell"),
]


def _db() -> sqlite3.Connection:
    first = not DB.exists()
    con = sqlite3.connect(DB)
    con.row_factory = sqlite3.Row
    if first:
        con.executescript("""
        CREATE TABLE products(id INTEGER PRIMARY KEY, name TEXT, category TEXT, price REAL, stock INTEGER, description TEXT);
        CREATE TABLE cart(user_id TEXT, product_id INTEGER, qty INTEGER, PRIMARY KEY(user_id, product_id));
        CREATE TABLE orders(id INTEGER PRIMARY KEY AUTOINCREMENT, user_id TEXT, total REAL, status TEXT, created TEXT DEFAULT CURRENT_TIMESTAMP);
        CREATE TABLE order_items(order_id INTEGER, product_id INTEGER, qty INTEGER, price REAL);""")
        con.executemany("INSERT INTO products(name,category,price,stock,description) VALUES(?,?,?,?,?)", PRODUCTS)
        con.commit()
    return con


@mcp.tool()
def search_products(query: str = "", category: str = "", max_price: float = 0) -> list[dict]:
    """Search the catalog. query matches name/description; category e.g. shoes, audio, wearables, outdoor, office,
    accessories, fitness; max_price 0 means no limit. Results sorted by price."""
    sql, args = "SELECT * FROM products WHERE stock > 0", []
    for w in [w for w in query.lower().split() if len(w) > 2]:
        sql += " AND lower(name||' '||description||' '||category) LIKE ?"
        args.append(f"%{w}%")
    if category:
        sql += " AND category = ?"
        args.append(category.lower())
    if max_price > 0:
        sql += " AND price <= ?"
        args.append(max_price)
    with _db() as con:
        return [dict(r) for r in con.execute(sql + " ORDER BY price LIMIT 8", args)]


@mcp.tool()
def get_product(product_id: int) -> dict:
    """Get full product details."""
    with _db() as con:
        r = con.execute("SELECT * FROM products WHERE id=?", (product_id,)).fetchone()
    return dict(r) if r else {"error": "product not found"}


@mcp.tool()
def add_to_cart(user_id: str, product_id: int, quantity: int = 1) -> dict:
    """Add a product to the user's cart (validates stock)."""
    if quantity < 1:
        return {"error": "quantity must be >= 1"}
    with _db() as con:
        p = con.execute("SELECT stock, name FROM products WHERE id=?", (product_id,)).fetchone()
        if not p:
            return {"error": "product not found"}
        cur = con.execute("SELECT qty FROM cart WHERE user_id=? AND product_id=?", (user_id, product_id)).fetchone()
        new_qty = quantity + (cur["qty"] if cur else 0)
        if new_qty > p["stock"]:
            return {"error": f"only {p['stock']} in stock"}
        con.execute("INSERT INTO cart VALUES(?,?,?) ON CONFLICT(user_id,product_id) DO UPDATE SET qty=?",
                    (user_id, product_id, new_qty, new_qty))
    return {"added": p["name"], "qty_in_cart": new_qty}


@mcp.tool()
def view_cart(user_id: str) -> dict:
    """Show the cart with line totals and the grand total."""
    with _db() as con:
        rows = con.execute("""SELECT p.id, p.name, p.price, c.qty, p.price*c.qty AS line_total
                              FROM cart c JOIN products p ON p.id=c.product_id WHERE c.user_id=?""", (user_id,)).fetchall()
    items = [dict(r) for r in rows]
    return {"items": items, "total": round(sum(i["line_total"] for i in items), 2)}


@mcp.tool()
def remove_from_cart(user_id: str, product_id: int) -> dict:
    """Remove a product from the cart."""
    with _db() as con:
        cur = con.execute("DELETE FROM cart WHERE user_id=? AND product_id=?", (user_id, product_id))
    return {"removed": cur.rowcount == 1}


@mcp.tool()
def checkout(user_id: str, confirmed: bool = False) -> dict:
    """Place the order for everything in the cart. confirmed must be true (user explicitly agreed to the total)."""
    if not confirmed:
        return {"error": "user has not confirmed"}
    con = _db()
    try:
        con.execute("BEGIN IMMEDIATE")  # atomic: stock check + decrement + order + clear cart
        items = con.execute("""SELECT p.id, p.name, p.price, p.stock, c.qty FROM cart c
                               JOIN products p ON p.id=c.product_id WHERE c.user_id=?""", (user_id,)).fetchall()
        if not items:
            con.rollback()
            return {"error": "cart is empty"}
        for i in items:
            if i["qty"] > i["stock"]:
                con.rollback()
                return {"error": f"insufficient stock for {i['name']}"}
        total = round(sum(i["price"] * i["qty"] for i in items), 2)
        oid = con.execute("INSERT INTO orders(user_id,total,status) VALUES(?,?,'processing')", (user_id, total)).lastrowid
        for i in items:
            con.execute("INSERT INTO order_items VALUES(?,?,?,?)", (oid, i["id"], i["qty"], i["price"]))
            con.execute("UPDATE products SET stock = stock - ? WHERE id=?", (i["qty"], i["id"]))
        con.execute("DELETE FROM cart WHERE user_id=?", (user_id,))
        con.commit()
        return {"order_id": oid, "total": total, "status": "processing"}
    except Exception as e:
        con.rollback()
        return {"error": f"checkout failed: {e}"}
    finally:
        con.close()


@mcp.tool()
def list_orders(user_id: str) -> list[dict]:
    """List the user's orders, newest first."""
    with _db() as con:
        return [dict(r) for r in con.execute("SELECT * FROM orders WHERE user_id=? ORDER BY id DESC LIMIT 10", (user_id,))]


@mcp.tool()
def get_order_status(user_id: str, order_id: int) -> dict:
    """Get an order's status and items (only the owner can see it)."""
    with _db() as con:
        o = con.execute("SELECT * FROM orders WHERE id=? AND user_id=?", (order_id, user_id)).fetchone()
        if not o:
            return {"error": "order not found"}
        items = con.execute("""SELECT p.name, i.qty, i.price FROM order_items i JOIN products p ON p.id=i.product_id
                               WHERE i.order_id=?""", (order_id,)).fetchall()
    return {**dict(o), "items": [dict(i) for i in items]}


@mcp.tool()
def cancel_order(user_id: str, order_id: int) -> dict:
    """Cancel an order that is still 'processing' and restock its items."""
    con = _db()
    try:
        con.execute("BEGIN IMMEDIATE")
        o = con.execute("SELECT status FROM orders WHERE id=? AND user_id=?", (order_id, user_id)).fetchone()
        if not o:
            con.rollback()
            return {"error": "order not found"}
        if o["status"] != "processing":
            con.rollback()
            return {"error": f"cannot cancel an order that is {o['status']}"}
        for i in con.execute("SELECT product_id, qty FROM order_items WHERE order_id=?", (order_id,)).fetchall():
            con.execute("UPDATE products SET stock = stock + ? WHERE id=?", (i["qty"], i["product_id"]))
        con.execute("UPDATE orders SET status='cancelled' WHERE id=?", (order_id,))
        con.commit()
        return {"order_id": order_id, "status": "cancelled"}
    finally:
        con.close()


if __name__ == "__main__":
    print("ecommerce MCP server starting", file=sys.stderr)
    mcp.run()

Overwriting e_ecommerce_agent/mcp_server.py


In [27]:
%%writefile e_ecommerce_agent/agent.py
"""(e) Production-style e-commerce agent: multi-agent + MCP + guardrails.

root shopping_agent (browse/cart/checkout)  --transfer-->  order_support_agent (status/cancel)
Both talk to a SQLite-backed MCP server (free replacement for AlloyDB + MCP Toolbox).
Guardrails (before_tool_callback): user_id is injected from session state (the LLM can't impersonate another
user) and checkout is blocked until the user explicitly confirms.
"""
import sys
from pathlib import Path

from google.adk.agents import LlmAgent
from google.adk.tools.mcp_tool.mcp_toolset import MCPToolset, StdioConnectionParams
from mcp import StdioServerParameters

from common import MODEL

SERVER = str(Path(__file__).with_name("mcp_server.py"))
INITIAL_STATE = {"user_id": "demo_user"}
USER_SCOPED = {"add_to_cart", "view_cart", "remove_from_cart", "checkout", "list_orders", "get_order_status", "cancel_order"}


def _toolset(names: list[str]) -> MCPToolset:
    return MCPToolset(
        connection_params=StdioConnectionParams(
            server_params=StdioServerParameters(command=sys.executable, args=[SERVER]), timeout=30),
        tool_filter=names,  # least privilege: each agent only sees the tools it needs
    )


shop_tools = _toolset(["search_products", "get_product", "add_to_cart", "view_cart", "remove_from_cart", "checkout"])
order_tools = _toolset(["list_orders", "get_order_status", "cancel_order"])
TOOLSETS = [shop_tools, order_tools]


def guardrails(tool, args, tool_context):
    """Runs before every tool call. Return a dict to block the call, or None to allow it."""
    if tool.name in USER_SCOPED:
        args["user_id"] = tool_context.state.get("user_id", "guest")  # never trust the LLM for identity
    if tool.name == "checkout" and not args.get("confirmed"):
        return {"error": "Checkout blocked: show the cart total, ask the user to confirm, then call checkout with confirmed=true."}
    return None


order_support_agent = LlmAgent(
    name="order_support_agent",
    model=MODEL,
    description="Handles order status, order history and cancellations.",
    instruction="""You are order support. Use list_orders / get_order_status / cancel_order.
Pass any user_id (it is overwritten server-side). Confirm before cancelling. Be brief and friendly.""",
    tools=[order_tools],
    before_tool_callback=guardrails,
)

root_agent = LlmAgent(
    name="shopping_agent",
    model=MODEL,
    description="Online store assistant: search products, manage cart, checkout.",
    instruction="""You are a friendly shopping assistant for an online store.
- Use search_products to find items; recommend with price and why it fits. Never invent products.
- Use add_to_cart when the user picks an item; show view_cart totals.
- Before checkout: show the cart + total and ask 'Shall I place this order?'. Only after an explicit yes
  call checkout with confirmed=true.
- For order status, history or cancellation, transfer to order_support_agent.
Pass any user_id (it is overwritten server-side).""",
    tools=[shop_tools],
    sub_agents=[order_support_agent],
    before_tool_callback=guardrails,
)

Overwriting e_ecommerce_agent/agent.py


In [28]:
!python -u run.py e 2>/dev/null


🧑 USER: I need running shoes under $120. What do you recommend?
------------------------------------------------------------
  🔧 [shopping_agent] calls search_products({'query': 'running', 'max_price': 120, 'category': 'shoes'})
  ✅ [shopping_agent] got search_products -> {'content': [{'type': 'text', 'text': '{\n  "id": 4,\n  "name": "CloudWalk Running Shoes",\n  "category": "shoes",\n  "price": 79.0,\n  "stock": 40,\n  "description": "Budget cushioned running shoe for beginners"\n}'}, { ...

🤖 [shopping_agent]
I found two great running shoes under $120 that match what you're looking for:

1. **CloudWalk Running Shoes** — **$79.00**
   - *Why it fits:* It's a budget-friendly, cushioned running shoe designed especially for beginners.
2. **AeroRun 2 Running Shoes** — **$99.00**
   - *Why it fits:* A lightweight daily trainer featuring breathable mesh for extra comfort on your runs.

Would you like to add either of these to your cart?


🧑 USER: Add the cheapest one to my cart, then show

## Summary
All five agents ran with the Gemini API free tier on Colab, with no paid services (SQLite instead of AlloyDB, local MCP servers, free web search).
Generated files are in `outputs/` and `workspace/`.